# SAE basics: encode prompts and inspect features

A **Sparse Autoencoder (SAE)** takes a language model's internal activations and breaks them into a list of features. Each feature roughly corresponds to one concept (a name, a topic, a grammatical role). For any given input, only a small fraction of features are active at a time, which makes the model's internal state easier to read.

This notebook walks through the simplest SAE workflow with Murano:

1. Load a pre-trained SAE from HuggingFace.
2. Encode a few prompts through it.
3. For each prompt, see which feature fires strongest and what concept it represents.

We use Gemma 2 2B with the gemma-scope SAE at layer 20.

**Install**:

```bash
pip install -e ".[sae,notebook]"
```

## Setup

Pick the model, the SAE release, and the SAE id. `SAEEncode` reads the target layer from the SAE's own config, so we don't have to specify it.

In [1]:
from murano import MuranoModel, Pipeline
from murano.steps import SAEEncode, SAEFeatureLabel, top_sae_features_per_prompt
from murano.steps.prompts import LoadPrompts

MODEL_ID = "google/gemma-2-2b-it"
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = "layer_20/width_16k/canonical"

## Load the model

`MuranoModel` wraps a HuggingFace model so the same step code works across Gemma, Llama, GPT-2, and other families.

In [2]:
model = MuranoModel(MODEL_ID)
model

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

MuranoModel('google/gemma-2-2b-it', layers=26, d=2304)

## Encode prompts through the SAE

A tiny pipeline: load prompts, then encode them through the SAE.

In [3]:
PROMPTS = [
    "The Eiffel Tower is located in the city of Paris",
    "The Colosseum is located in the city of Rome",
    "The Brandenburg Gate is located in the city of Berlin",
    "Senso-ji temple is located in the city of Tokyo",
    "Red Square is located in the city of Moscow",
]

results = Pipeline([
    LoadPrompts(PROMPTS),
    SAEEncode(model, release=SAE_RELEASE, sae_id=SAE_ID),
]).run()

record = results["sae_record"]
print(f"Layer: {record.layer}")
print(f"SAE width: {record.n_features}")
print(f"Activations shape: {tuple(record.activations.shape)}")

Layer: 20
SAE width: 16384
Activations shape: (5, 13, 16384)


`record.activations` has shape `[N, seq, n_features]`: one SAE code vector per token. SAEs are sparse: even though hundreds of features may fire on a token, that is still a small fraction of the full width (e.g. a few hundred out of 16,384), so each code vector is mostly zeros. Below we count how many fire per token.

In [4]:
active_per_token = (record.activations > 0).sum(dim=-1)
# attention_mask.bool() keeps only real, non-padding token positions.
mean_active = active_per_token[record.attention_mask.bool()].float().mean().item()
print(f"Average active features per real token: {mean_active:.1f} / {record.n_features}")

Average active features per real token: 750.5 / 16384


## What is each sentence about?

Different prompts activate different features, which is the whole point of an SAE. To read what a sentence is about *as a whole*, we average each feature's activation across its content tokens and take the top one.

There is a catch: a few features fire on almost every token (high-frequency, mostly uninterpretable) and would dominate any average. `top_sae_features_per_prompt(record, reduce="mean")` drops those automatically: across a diverse batch, a real concept feature fires on only its own prompt and stands out, while broad features fire everywhere and are filtered. The five prompts above are deliberately varied (different cities) so each concept feature is easy to isolate.

(Use the default `reduce="last"` instead to read only the last token, i.e. what the model is about to predict next; for "...located in Paris" that is usually a generic "city" feature.)

In [5]:
top_feats = top_sae_features_per_prompt(record, n=1, reduce="mean")
for prompt, feats in zip(record.texts, top_feats):
    print(f"{prompt!r}")
    print(f"  top feature: {feats}")
    print()

'The Eiffel Tower is located in the city of Paris'
  top feature: [5516]

'The Colosseum is located in the city of Rome'
  top feature: [1780]

'The Brandenburg Gate is located in the city of Berlin'
  top feature: [16173]

'Senso-ji temple is located in the city of Tokyo'
  top feature: [1315]

'Red Square is located in the city of Moscow'
  top feature: [1116]



## What does each feature mean?

A feature ID is just a number. To learn what it represents, `SAEFeatureLabel` reads the feature's direction in the model's hidden-state space and projects it through the model's unembedding (the layer that turns hidden states into output-token logits). The top token there is the one this feature most pushes the model to say. We call it the token the feature *promotes*.

Each prompt is about a landmark in a different country, and the top feature for each should promote that country's nationality (French, Italian, German, and so on). That is the SAE doing its job: it has decomposed each sentence into a clean, interpretable concept.

In [6]:
all_feat_ids = sorted({fid for feats in top_feats for fid in feats})

results = Pipeline(
    [SAEFeatureLabel(model, feat_ids=all_feat_ids, k_tokens=1)]
).run(results)

labels = results["feature_labels"]
for prompt, feats in zip(record.texts, top_feats):
    fid = feats[0]
    token = labels.tokens[fid][0]
    logit = labels.logits[fid][0]
    print(f"{prompt!r}")
    print(f"  feat {fid} promotes {token!r} (logit {logit:.2f})")
    print()

'The Eiffel Tower is located in the city of Paris'
  feat 5516 promotes ' French' (logit 52.25)

'The Colosseum is located in the city of Rome'
  feat 1780 promotes ' Italian' (logit 56.00)

'The Brandenburg Gate is located in the city of Berlin'
  feat 16173 promotes ' German' (logit 39.00)

'Senso-ji temple is located in the city of Tokyo'
  feat 1315 promotes ' Japan' (logit 60.25)

'Red Square is located in the city of Moscow'
  feat 1116 promotes ' Russian' (logit 85.50)



## What's next

Notebook [`02_feature_steering.ipynb`](02_feature_steering.ipynb) shows how to find a feature for a specific concept and steer the model's generations with it.